In [17]:
import os
import sys

import numpy as np
import pandas as pd
from Bio import SeqIO

sys.path.append("./additional_code")
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../../data/our_data/"

/home/hanxd/Repositories/ESP/compare/code


In [18]:
querynamelist = [
    "Arabidopsis_thaliana",
    "Erigeron_breviscapus",
    "Glycine_max",
    "Zea_mays",
    "sheetall",
]

In [ ]:
from rdkit import Chem


def sdf2smile(path):
    suppl = Chem.SDMolSupplier(path)
    for mol in suppl:
        if mol is not None:
            smiles = Chem.MolToSmiles(mol)
            return smiles

In [ ]:
for queryname in querynamelist:
    result = pd.read_csv(f"../output/{queryname}.csv")
    enzymes = result["enzyme"].unique()
    substrates = result["substrate"].unique()
    seqold = SeqIO.to_dict(
        SeqIO.parse(f"{our_data}/screening1/FunP450_All_06_new_Plant_q.fasta", "fasta")
    )
    if queryname == "sheetall":
        seqs = seqold
    else:
        seqs = SeqIO.to_dict(
            SeqIO.parse(f"{our_data}/screening2/{queryname}_q.pep", "fasta")
        )
    for enzyme in enzymes:
        try:
            result.loc[result["enzyme"] == enzyme, "seq"] = str(seqs[enzyme].seq)
        except:
            print("matched in before:", enzyme)
            result.loc[result["enzyme"] == enzyme, "seq"] = str(seqold[enzyme].seq)
    for substrate in substrates:
        result.loc[result["substrate"] == substrate, "smile"] = sdf2smile(
            f"{our_data}/SDF_399/{substrate}.sdf"
        )
    result.to_csv(f"../output/{queryname}_ss.csv", index=False)
    resultnocc = result.drop_duplicates(subset=["enzyme", "substrate"])
    resultnocc.to_csv(f"../output/{queryname}_ss_nocc.csv", index=False)
    resultnocc4transformer = resultnocc[["smile", "seq"]]
    resultnocc4transformer = resultnocc4transformer.rename(
        columns={"smile": "Compound", "seq": "Protein"}
    )
    resultnocc4transformer.to_csv(f"../output/{queryname}_4predict.csv", index=False)

matched in before: CYP93E1
matched in before: CYP93B16
matched in before: CYP71D9
matched in before: CYP93A1
matched in before: CYP71A10
matched in before: CYP93C1
matched in before: CYP81E100
matched in before: CYP72A69
matched in before: CYP82D26
matched in before: CYP701A43
matched in before: CYP81A37
matched in before: CYP81A38


In [21]:
result["substrate"].nunique()

144

In [22]:
len(result)

40775